# Fine-mapping 
## Description

This notebook fine-maps cis-regions on the toy example dataset using three
complementary methods from the xQTL protocol:

- **Stage 1 — SuSiE TWAS** (`mnm_regression.ipynb susie_twas`): individual-level
  fine-mapping + TWAS weights, using **in-sample LD** computed directly from the
  study genotypes.
- **Stage 2 — RSS (summary-statistics fine-mapping)** (`rss_analysis.ipynb univariate_rss`):
  fine-maps a GWAS region using an **external LD reference (LD sketch)**.
- **Stage 3 — fSuSiE** (`mnm_regression.ipynb fsusie`): functional SuSiE
  fine-mapping, also using in-sample LD.

It consumes the genotype, phenotype and covariate outputs produced by the
upstream user-friendly notebooks: `2_genotype_preprocessing.ipynb`,
`3_genotype_pca.ipynb`, `4_covariates_preprocessing.ipynb`,
and the per-chromosome cis phenotype list from `1_phenotype_preprocessing.ipynb`.


## A note on LD references in this toy example

`susie_twas` can use **either** in-sample LD **or** the external LD sketch. By default it uses **in-sample LD** (computed from the study genotypes), but if you pass `--ld_reference_meta_file` its fine-mapping will instead load LD from that sketch (see Step 1). `fsusie` only uses **in-sample LD**; it has no external-LD option.

`univariate_rss` works from GWAS **summary statistics** and therefore *requires* an external LD reference. This toy example ships a small, **self-contained LD sketch** built from the same 60-sample genotype with `pipeline/rss_ld_sketch.ipynb`. It lives under `input/ld_reference/` and is referenced through `input/ld_reference/protocol_example.ld_meta_file.tsv`.

The sketch covers **all autosomes (chr1-chr22)** as per-chromosome PLINK2 **pgen** triplets (`.pgen`/`.pvar`/`.psam`, plus a `.afreq`), one block directory per chromosome. The meta file has columns `#chr start end path`, where `path` is the pgen **prefix** (no extension) resolved relative to the meta file own directory. `pecotmr` `load_LD_matrix` auto-detects the pgen format and reconstructs the LD on the fly. In a full analysis you would point `--ld-meta-data` at a population-scale reference (e.g. ADSP EUR); the sketch demonstrates the identical interface on a small, portable file.


## Input Files

| File | Produced by | Used as |
|------|-------------|---------|
| `output/genotype_by_chrom/protocol_example.genotype.merged.plink_qc.genotype_by_chrom_files.txt` | `2_genotype_preprocessing.ipynb` | `--genoFile` (per-chrom list) |
| `output/phenotype/phenotype_by_chrom_for_cis/bulk_rnaseq.phenotype_by_chrom_files.region_list.txt` | `1_phenotype_preprocessing.ipynb` | `--phenoFile` |
| `output/covariate/*.Marchenko_PC.gz` | `4_covariates_preprocessing.ipynb` | `--covFile` |
| `reference_data/TAD/TADB_enhanced_cis.bed` | reference data | `--customized-association-windows` |
| `input/ld_reference/protocol_example.ld_meta_file.tsv` | `rss_ld_sketch.ipynb` (LD sketch) | `--ld-meta-data` (RSS) |
| `data/gwas_meta_data.txt` | reference GWAS sumstats | `--gwas-meta-data` (RSS) |


## Steps

### **Step 1.** SuSiE TWAS fine-mapping (in-sample LD or LD sketch)

Runs `mnm_regression.ipynb susie_twas` on a single example gene (`ENSG00000130538`, chr22).

The optional `--ld_reference_meta_file` controls the LD used by **both** the fine-mapping and the TWAS-weights stages. When you pass it (as below, pointing at the bundled all-chromosome LD sketch `input/ld_reference/protocol_example.ld_meta_file.tsv`), fine-mapping loads LD from that sketch via `pecotmr load_LD_matrix`. When you omit it, the step falls back to **in-sample LD** computed from the study genotypes themselves. The default in this example uses the sketch so both stages share one consistent LD source.

This produces a fine-mapping result (`*.univariate_bvsr.rds`) and TWAS weights (`*.univariate_twas_weights.rds`).


In [ ]:
sos run pipeline/mnm_regression.ipynb susie_twas \
    --name protocol_example_susie_twas \
    --genoFile output/genotype_by_chrom/protocol_example.genotype.merged.plink_qc.genotype_by_chrom_files.txt \
    --phenoFile output/phenotype/phenotype_by_chrom_for_cis/bulk_rnaseq.phenotype_by_chrom_files.region_list.txt \
    --covFile $(ls output/covariate/*Marchenko_PC.gz) \
    --customized-association-windows reference_data/TAD/TADB_enhanced_cis.bed \
    --phenotype-names bulk_rnaseq \
    --max-cv-variants 5000 \
    --ld_reference_meta_file input/ld_reference/protocol_example.ld_meta_file.tsv \
    --region-name ENSG00000130538 \
    --save-data \
    --cwd output/finemapping/susie_twas \
    -s force


### **Step 2.** RSS fine-mapping (summary statistics + LD sketch)

Runs `rss_analysis.ipynb univariate_rss` on a GWAS region
(`22:49355984-50799822`) that overlaps the LD sketch. The LD reference is
supplied through `input/ld_reference/protocol_example.ld_meta_file.tsv`; the GWAS summary
statistics are described in `data/gwas_meta_data.txt`. This produces
`*.univariate_susie_rss.rds`.


In [ ]:
sos run pipeline/rss_analysis.ipynb univariate_rss \
    --ld-meta-data input/ld_reference/protocol_example.ld_meta_file.tsv \
    --gwas-meta-data data/gwas_meta_data.txt \
    --qc-method "rss_qc" \
    --impute \
    --finemapping_method "susie_rss" \
    --skip-analysis-pip-cutoff 0 \
    --region-name 22:49355984-50799822 \
    --cwd output/finemapping/rss_analysis \
    -s force


### **Step 3.** fSuSiE fine-mapping (in-sample LD)

Runs `mnm_regression.ipynb fsusie` (functional SuSiE) on the same example gene
(`ENSG00000130538`). Like `susie_twas`, it uses in-sample LD and needs no
external LD reference. This produces `*.fsusie_mixture_normal_TI__top_pc_weights.rds`.


In [ ]:
sos run pipeline/mnm_regression.ipynb fsusie \
    --name protocol_example_fsusie \
    --genoFile output/genotype_by_chrom/protocol_example.genotype.merged.plink_qc.genotype_by_chrom_files.txt \
    --phenoFile output/phenotype/phenotype_by_chrom_for_cis/bulk_rnaseq.phenotype_by_chrom_files.region_list.txt \
    --covFile $(ls output/covariate/*Marchenko_PC.gz) \
    --customized-association-windows reference_data/TAD/TADB_enhanced_cis.bed \
    --phenotype-names bulk_rnaseq \
    --save-data \
    --region-name ENSG00000130538 \
    --cwd output/finemapping/fsusie \
    -s force


## Output Files

| File | Stage |
|------|-------|
| `output/finemapping/susie_twas/fine_mapping/protocol_example_susie_twas.chr22_ENSG00000130538.univariate_bvsr.rds` | Step 1 |
| `output/finemapping/susie_twas/twas_weights/protocol_example_susie_twas.chr22_ENSG00000130538.univariate_twas_weights.rds` | Step 1 |
| `output/finemapping/rss_analysis/univariate_rss/RSS_QC_RAISS_imputed.chr22_49355984_50799822.univariate_susie_rss.rds` | Step 2 |
| `output/finemapping/fsusie/fsus/protocol_example_fsusie.chr22_15528191_15529138.fsusie_mixture_normal_TI__top_pc_weights.rds` | Step 3 |


## Anticipated Results

Each stage writes an `.rds` result object for the analyzed region. On this
minimal toy dataset the credible sets and posterior inclusion probabilities are
not biologically meaningful; the goal is to confirm that all three fine-mapping
pipelines run end-to-end and emit their expected output objects.
